In [1]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    classification_report,
)


In [2]:
def load_and_preprocess_data(data_path, label_column_name, sample_fraction=0.1):
    # Load all CSV files from the directory
    all_files = glob.glob(os.path.join(data_path, '*.csv'))

    df_list = []
    print("Loading CSV files...")
    for filename in all_files:
        temp_df = pd.read_csv(filename)
        df_list.append(temp_df)

    df = pd.concat(df_list, ignore_index=True)

    # Clean column names
    df.columns = df.columns.str.strip()

    # Check and update label column name if necessary
    if label_column_name not in df.columns:
        possible_label_columns = [col for col in df.columns if 'label' in col.lower()]
        if possible_label_columns:
            label_column_name = possible_label_columns[0]
            print(f"Using '{label_column_name}' as the label column.")
        else:
            raise KeyError(f"Label column '{label_column_name}' not found in dataframe.")

    # Map labels to 'Benign' and 'Malicious'
    df[label_column_name] = np.where(df[label_column_name] == 'BENIGN', 'Benign', 'Malicious')

    # Stratified sampling to maintain class distribution
    df = df.groupby(label_column_name, group_keys=False).apply(
        lambda x: x.sample(frac=sample_fraction, random_state=42)
    ).reset_index(drop=True)

    # Separate features and labels
    X = df.drop(columns=[label_column_name])
    y = df[label_column_name]

    return X, y


In [3]:
def preprocess_and_split(X, y, test_size=0.2, random_state=42):
    # Convert categorical features to numeric if any
    X = pd.get_dummies(X)

    # Identify numeric columns
    numeric_cols = X.select_dtypes(include=['float64', 'int64', 'uint8']).columns

    # Handle infinite values
    X.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Impute missing values using median
    imputer = SimpleImputer(strategy='median')
    X[numeric_cols] = imputer.fit_transform(X[numeric_cols])

    # Scale features using Min-Max Scaler
    scaler = MinMaxScaler()
    X[numeric_cols] = scaler.fit_transform(X[numeric_cols])

    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    # Split data with stratification
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=test_size, random_state=random_state, stratify=y_encoded
    )

    return X_train, X_test, y_train, y_test, label_encoder.classes_


In [4]:
def correlation_selection(X, y, feature_number=10):
    # Combine X and y into a single DataFrame
    df = X.copy()
    df['Target'] = y

    # Calculate correlation matrix
    corr_matrix = df.corr()

    # Get correlation with target variable
    corr_with_target = corr_matrix['Target'].drop('Target')

    # Select top features based on absolute correlation
    top_features = corr_with_target.abs().sort_values(ascending=False).head(feature_number).index.tolist()

    print(f"Top {feature_number} features selected by correlation:")
    print(top_features)

    return top_features


In [5]:
def univariate_selection(X, y, feature_number=10):
    selector = SelectKBest(score_func=f_classif, k=feature_number)
    selector.fit(X, y)

    selected_indices = selector.get_support(indices=True)
    selected_features = X.columns[selected_indices].tolist()

    print(f"Top {feature_number} features selected by univariate selection:")
    print(selected_features)

    return selected_features


In [6]:
def random_forest_selection(X, y, feature_number=10):
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)

    importances = model.feature_importances_
    feature_importance = pd.Series(importances, index=X.columns)
    top_features = feature_importance.sort_values(ascending=False).head(feature_number).index.tolist()

    print(f"Top {feature_number} features selected by Random Forest importance:")
    print(top_features)

    return top_features


In [7]:
def extra_trees_selection(X, y, feature_number=10):
    model = ExtraTreesClassifier(n_estimators=100, random_state=42)
    model.fit(X, y)

    importances = model.feature_importances_
    feature_importance = pd.Series(importances, index=X.columns)
    top_features = feature_importance.sort_values(ascending=False).head(feature_number).index.tolist()

    print(f"Top {feature_number} features selected by Extra Trees importance:")
    print(top_features)

    return top_features


In [8]:
def train_and_evaluate_model(X_train, X_test, y_train, y_test, class_names):
    # Initialize the model (using Decision Tree for quick training)
    model = DecisionTreeClassifier(random_state=42)

    print("\nTraining the model...")
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate performance metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    print("\nModel Performance:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion Matrix
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

    # ROC Curve and AUC
    if len(class_names) == 2:
        y_probs = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_probs)
        fpr, tpr, thresholds = roc_curve(y_test, y_probs)
        plt.plot(fpr, tpr, label=f'AUC = {auc:.4f}')
        plt.plot([0, 1], [0, 1], 'r--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.show()


In [9]:
def main():
    data_path = r'C:\Users\joshb\OneDrive\Documents\Final Project\CICDS 2017\MachineLearningCVE'
    label_column_name = 'Label'  # Update if necessary
    sample_fraction = 1.0  # Adjust sample fraction for quick testing

    # Step 1: Load and preprocess data
    print("Loading and preprocessing data...")
    X, y = load_and_preprocess_data(data_path, label_column_name, sample_fraction)

    # Step 2: Preprocess and split data
    print("Preprocessing and splitting data...")
    X_train, X_test, y_train, y_test, class_names = preprocess_and_split(X, y)

    # Step 3: Feature Selection
    print("Applying feature selection...")
    # Choose one of the feature selection methods
    # For example, using Random Forest feature importance
    selected_features = random_forest_selection(X_train, y_train, feature_number=10)

    # Reduce datasets to selected features
    X_train_selected = X_train[selected_features]
    X_test_selected = X_test[selected_features]

    # Step 4: Train and evaluate the model
    train_and_evaluate_model(X_train_selected, X_test_selected, y_train, y_test, class_names)


In [ ]:
if __name__ == "__main__":
    main()
